In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.DataFrame({
    "Condition": ["A", "a", "cond1", "B", "b"],
    "Gender": ["male", "Female", "female", "Male", "male"]
})

String methods (.str accessor) operate element-wise but are internally optimized.
> str.upper() prevents artificial group splitting ("A" vs "a").

In [3]:
# Standardizes capitalization:
df["Condition"] = df["Condition"].str.upper()
print(df)

  Condition  Gender
0         A    male
1         A  Female
2     COND1  female
3         B    Male
4         B    male


> str.contains() filters subsets.

In [4]:
# Detects patterns (regex-supported):
df["is_cond1"] = df["Condition"].str.contains("COND1", case=False)
print(df)

  Condition  Gender  is_cond1
0         A    male     False
1         A  Female     False
2     COND1  female      True
3         B    Male     False
4         B    male     False


> str.replace() standardizes inconsistent labels.

In [6]:
df["Condition"] = df["Condition"].str.replace("COND1", "CONDITION1", regex=False) # regex=False to avoid unintended pattern matching.
print(df)

    Condition  Gender  is_cond1
0           A    male     False
1           A  Female     False
2  CONDITION1  female      True
3           B    Male     False
4           B    male     False


#### Categorical dtype
By default, strings are stored as object. This is memory-inefficient and semantically ambiguous.   
It matters for:
- Memory efficiency
- Explicit levels
- Statistical modeling
- Ordered comparisons

In [9]:
# Convert to categorical:
df["Gender"] = df["Gender"].astype("category")

##### Ordered categorical variables

In [17]:
df["Condition"] = df["Condition"].astype("category")

df["Condition"] = df["Condition"].cat.set_categories(
    ["A", "B", "CONDITION1"], # Conditions often have theoretical order.
    ordered=True
)

# Note: without ordering, plots may appear alphabetically, distorting interpretation.

print("Ready for plotting:\n", df.sort_values("Condition"))

Ready for plotting:
     Condition  Gender  is_cond1
0           A    male     False
1           A  Female     False
3           B    Male     False
4           B    male     False
2  CONDITION1  female      True


##### Factorization
Factorization converts unique categories to integer codes.

In [15]:
codes, uniques = pd.factorize(df["Condition"])
# codes: numeric representation
# uniques: category mapping

# Equivalent with categorical:
print(df["Condition"].cat.codes)

0    0
1    0
2    2
3    1
4    1
dtype: int8


Note: this is useful for modeling pipelines requiring numeric inputs, but it is not equivalent to meaningful ordinal coding unless order is specified.

#### Why Proper Categorical Types Matter for Statistics

Using object strings instead of categorical dtype introduces several risks:
- Accidental level proliferation.
- Alphabetical ordering instead of theoretical ordering.
- Incorrect contrast coding in regression models.
- Memory inefficiency in large datasets.
- Silent coercion when exporting to modeling frameworks.

Categorical dtype formalizes group structure. It makes the researcher declare what the levels are and whether order matters.  
This reduces analytic ambiguity.